# DTM: Temporal Topic Analysis (Native)

DTM is **natively temporal** — uses `get_topic_word_dist(topic, timepoint)` directly.

In [1]:
import ast
import time
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
import tomotopy as tp
import warnings
warnings.filterwarnings("ignore")

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]
BASE_DIR = Path("../../../../data/preprocess")
MODEL_DIR = Path("../../../../models/dtm/tuning")
RESULT_DIR = Path("../../../../results/dtm/temporal")
VERSION = "v1"
TOP_N_WORDS = 10
RBO_P = 0.9

for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

In [3]:
def get_topic_words_at_timepoint(model, timepoint, top_n=10):
    vocab = list(model.used_vocabs)
    result = {}
    for tid in range(model.k):
        dist = model.get_topic_word_dist(tid, timepoint=timepoint)
        top_idx = np.argsort(dist)[::-1][:top_n]
        result[tid] = [vocab[i] for i in top_idx]
    return result


def get_aggregated_topic_words(model, top_n=10):
    vocab = list(model.used_vocabs)
    result = {}
    for tid in range(model.k):
        avg_dist = np.zeros(len(vocab))
        for t in range(model.num_timepoints):
            avg_dist += np.array(model.get_topic_word_dist(tid, timepoint=t))
        avg_dist /= model.num_timepoints
        top_idx = np.argsort(avg_dist)[::-1][:top_n]
        result[tid] = [vocab[i] for i in top_idx]
    return result


def rbo(list_1, list_2, p=0.9):
    k = min(len(list_1), len(list_2))
    if k == 0:
        return 0.0
    score = 0.0
    for d in range(1, k + 1):
        agreement = len(set(list_1[:d]) & set(list_2[:d])) / d
        score += (p ** (d - 1)) * agreement
    return score * (1 - p)


def calculate_irbo(topics_words_list, p=0.9):
    if len(topics_words_list) < 2:
        return 0.0
    scores = [1.0 - rbo(topics_words_list[i], topics_words_list[j], p)
              for i, j in combinations(range(len(topics_words_list)), 2)]
    return np.mean(scores)

## Load Models & Data

In [4]:
all_models = {}
all_data = {}
all_years = {}
all_year_to_tp = {}

for subject in LIST_SUBJECT:
    print(f"\nLoading {subject}...")

    model = tp.DTModel.load(str(MODEL_DIR / subject / "best_model.bin"))
    all_models[subject] = model

    df = pd.read_csv(BASE_DIR / subject / "bow" / f"{VERSION}.csv")
    df["submitted_date"] = pd.to_datetime(df["submitted_date"])
    df["year"] = df["submitted_date"].dt.year

    # Filter empty docs (DTM skips them during training)
    df["tokens"] = df["text"].apply(lambda x: ast.literal_eval(x))
    df = df[df["tokens"].apply(len) > 0].reset_index(drop=True)

    years = sorted(df["year"].unique())
    year_to_tp = {year: i for i, year in enumerate(years)}
    all_years[subject] = years
    all_year_to_tp[subject] = year_to_tp

    print(f"  Model docs: {len(model.docs)}, DataFrame: {len(df)}")
    topics = [int(np.argmax(doc.get_topic_dist())) for doc in model.docs]
    df["topic"] = topics

    all_data[subject] = df
    print(f"  {subject}: {len(df):,} docs, {model.k} topics, "
          f"{len(years)} years ({years[0]}-{years[-1]})")

print(f"\n✅ All subjects loaded")


Loading cs...
  Model docs: 165756, DataFrame: 165756
  cs: 165,756 docs, 50 topics, 26 years (2000-2025)

Loading math...
  Model docs: 157084, DataFrame: 157084
  math: 157,084 docs, 50 topics, 26 years (2000-2025)

Loading physics...
  Model docs: 146311, DataFrame: 146311
  physics: 146,311 docs, 60 topics, 26 years (2000-2025)

✅ All subjects loaded


## Topic Prevalence Over Time

In [5]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    years = all_years[subject]
    global_tw = get_aggregated_topic_words(model, top_n=5)

    rows = []
    for year in years:
        year_df = df[df["year"] == year]
        counts = year_df["topic"].value_counts()
        for tid in range(model.k):
            count = counts.get(tid, 0)
            rows.append({"subject": subject, "year": year, "topic_id": tid,
                         "doc_count": count, "total_docs_year": len(year_df),
                         "proportion": round(count / len(year_df), 6) if len(year_df) > 0 else 0,
                         "top_words": ", ".join(global_tw[tid])})

    pd.DataFrame(rows).to_csv(RESULT_DIR / subject / "topic_prevalence.csv", index=False)
    print(f"  {subject.upper()}: Saved topic_prevalence.csv")

  CS: Saved topic_prevalence.csv
  MATH: Saved topic_prevalence.csv
  PHYSICS: Saved topic_prevalence.csv


## Topic Word Evolution (Native DTM)

In [6]:
all_topic_words_per_year = {}

for subject in LIST_SUBJECT:
    model = all_models[subject]
    years = all_years[subject]
    year_to_tp = all_year_to_tp[subject]

    topic_words_per_year = {}
    rows = []
    for year in years:
        tw = get_topic_words_at_timepoint(model, timepoint=year_to_tp[year], top_n=TOP_N_WORDS)
        for tid, words in tw.items():
            topic_words_per_year[(year, tid)] = words
            rows.append({"subject": subject, "year": year,
                         "topic_id": tid, "top_words": ", ".join(words)})

    all_topic_words_per_year[subject] = topic_words_per_year
    pd.DataFrame(rows).to_csv(RESULT_DIR / subject / "topic_word_evolution.csv", index=False)
    print(f"  {subject.upper()}: Saved topic_word_evolution.csv ({len(rows)} rows)")

  CS: Saved topic_word_evolution.csv (1300 rows)
  MATH: Saved topic_word_evolution.csv (1300 rows)
  PHYSICS: Saved topic_word_evolution.csv (1560 rows)


## Per-Year Coherence & IRBO

In [7]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    years = all_years[subject]
    topic_words_per_year = all_topic_words_per_year[subject]

    print(f"\n{'='*70}")
    print(f"Per-Year Metrics: {subject.upper()} ({model.k} topics)")
    print(f"{'='*70}")

    rows = []
    for year in years:
        year_df = df[df["year"] == year]
        # Use pre-parsed tokens column (avoid re-parsing)
        texts = year_df["tokens"].tolist()
        dictionary = Dictionary(texts)

        active_topics = sorted(year_df["topic"].unique())
        year_tw = [topic_words_per_year[(year, tid)] for tid in active_topics
                   if (year, tid) in topic_words_per_year]

        valid_tw = [tw for tw in year_tw if len(tw) >= 2]
        if valid_tw:
            cm = CoherenceModel(topics=valid_tw, texts=texts,
                                dictionary=dictionary, coherence='c_v', processes=1)
            coherence = cm.get_coherence()
        else:
            coherence = 0.0

        irbo = calculate_irbo(year_tw, p=RBO_P)
        quality = 2 * coherence * irbo / (coherence + irbo) if (coherence + irbo) > 0 else 0.0

        print(f"  {year}: {len(year_df):>6,} docs | {len(active_topics):>3} topics | "
              f"Q={quality:.4f} (C={coherence:.4f}, IRBO={irbo:.4f})")

        rows.append({"subject": subject, "year": year, "num_docs": len(year_df),
                     "num_topics_total": model.k, "num_topics_active": len(active_topics),
                     "coherence_cv": round(coherence, 6), "irbo_mean": round(irbo, 6),
                     "topic_quality": round(quality, 6)})

    pd.DataFrame(rows).to_csv(RESULT_DIR / subject / "per_year_metrics.csv", index=False)
    print(f"  Saved: per_year_metrics.csv")


Per-Year Metrics: CS (50 topics)
  2000:    488 docs |  49 topics | Q=0.4835 (C=0.3674, IRBO=0.7069)
  2001:    594 docs |  50 topics | Q=0.4234 (C=0.2869, IRBO=0.8075)
  2002:    648 docs |  50 topics | Q=0.4214 (C=0.2954, IRBO=0.7354)
  2003:    825 docs |  50 topics | Q=0.4184 (C=0.2905, IRBO=0.7476)
  2004:    948 docs |  50 topics | Q=0.4062 (C=0.2869, IRBO=0.6955)
  2005:  1,000 docs |  50 topics | Q=0.4681 (C=0.3566, IRBO=0.6808)
  2006:  1,000 docs |  50 topics | Q=0.4883 (C=0.3835, IRBO=0.6718)
  2007:  1,000 docs |  50 topics | Q=0.5232 (C=0.4176, IRBO=0.7005)
  2008:  1,000 docs |  50 topics | Q=0.5043 (C=0.3732, IRBO=0.7773)
  2009:  1,000 docs |  50 topics | Q=0.4958 (C=0.3635, IRBO=0.7798)
  2010:  1,362 docs |  50 topics | Q=0.4983 (C=0.3615, IRBO=0.8016)
  2011:  1,622 docs |  50 topics | Q=0.5092 (C=0.3696, IRBO=0.8185)
  2012:  2,254 docs |  50 topics | Q=0.4979 (C=0.3542, IRBO=0.8374)
  2013:  2,719 docs |  50 topics | Q=0.5335 (C=0.3873, IRBO=0.8568)
  2014:  2,989

## Topic Trends

In [8]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    global_tw = get_aggregated_topic_words(model, top_n=5)

    rows = []
    for tid in range(model.k):
        topic_df = df[df["topic"] == tid]
        if len(topic_df) == 0:
            continue

        year_counts = topic_df["year"].value_counts().sort_index()
        total_per_year = df["year"].value_counts().sort_index()
        proportions = (year_counts / total_per_year).fillna(0)

        topic_years = sorted(year_counts.index)
        early_mean = proportions[topic_years[:5]].mean()
        late_mean = proportions[topic_years[-5:]].mean()

        if early_mean < 1e-6 and late_mean < 1e-6:
            trend_ratio = 1.0
        else:
            trend_ratio = late_mean / max(early_mean, 1e-4)

        trend_label = "GROWING" if trend_ratio > 2.0 else ("DECLINING" if trend_ratio < 0.5 else "STABLE")

        rows.append({"subject": subject, "topic_id": tid,
                     "top_words": ", ".join(global_tw[tid]),
                     "total_docs": len(topic_df),
                     "first_year": year_counts.index.min(), "last_year": year_counts.index.max(),
                     "early_proportion": round(early_mean, 6),
                     "late_proportion": round(late_mean, 6),
                     "trend_ratio": round(trend_ratio, 4), "trend": trend_label})

    trends_df = pd.DataFrame(rows)
    trends_df.to_csv(RESULT_DIR / subject / "topic_trends.csv", index=False)

    g = len(trends_df[trends_df["trend"] == "GROWING"])
    s = len(trends_df[trends_df["trend"] == "STABLE"])
    d = len(trends_df[trends_df["trend"] == "DECLINING"])
    print(f"  {subject.upper()}: Growing={g}, Stable={s}, Declining={d}")

  CS: Growing=6, Stable=39, Declining=5
  MATH: Growing=7, Stable=33, Declining=10
  PHYSICS: Growing=4, Stable=46, Declining=10


## Top 5 Growing & Declining Topics

In [9]:
for subject in LIST_SUBJECT:
    trends_df = pd.read_csv(RESULT_DIR / subject / "topic_trends.csv")

    print(f"\n{'='*80}")
    print(f"  {subject.upper()}")
    print(f"{'='*80}")

    growing = trends_df[trends_df["trend"] == "GROWING"].sort_values("trend_ratio", ascending=False)
    declining = trends_df[trends_df["trend"] == "DECLINING"].sort_values("trend_ratio", ascending=True)

    print(f"\n  📈 TOP 5 GROWING:")
    for _, row in growing.head(5).iterrows():
        print(f"    T{int(row['topic_id']):>3} | {row['trend_ratio']:>7.2f}x | "
              f"{row['early_proportion']:.4f} → {row['late_proportion']:.4f} | {row['top_words']}")

    print(f"\n  📉 TOP 5 DECLINING:")
    for _, row in declining.head(5).iterrows():
        print(f"    T{int(row['topic_id']):>3} | {row['trend_ratio']:>7.4f}x | "
              f"{row['early_proportion']:.4f} → {row['late_proportion']:.4f} | {row['top_words']}")


  CS

  📈 TOP 5 GROWING:
    T 23 |    2.91x | 0.0174 → 0.0505 | image, algorithm, node, semantic, selection
    T 38 |    2.40x | 0.0103 → 0.0248 | strategy, prediction, robustness, network, generative
    T 28 |    2.24x | 0.0146 → 0.0327 | information, deep_learning, network, standard, sequence
    T  9 |    2.10x | 0.0265 → 0.0557 | dataset, code, accuracy, object, language
    T 31 |    2.02x | 0.0127 → 0.0255 | datum, deep, machine, prior, policy

  📉 TOP 5 DECLINING:
    T  1 |  0.3359x | 0.0455 → 0.0153 | network, scale, setting, knowledge, semantic
    T  7 |  0.3760x | 0.0301 → 0.0113 | dataset, end, distribution, property, crucial
    T 36 |  0.4192x | 0.0159 → 0.0067 | visual, audio, particular, area, assessment
    T 20 |  0.4586x | 0.0187 → 0.0086 | effectiveness, network, property, limitation, non
    T  2 |  0.4799x | 0.0392 → 0.0188 | point, capability, human, cost, output

  MATH

  📈 TOP 5 GROWING:
    T 48 |    2.95x | 0.0098 → 0.0288 | measure, theory, optimal, op

## Evolution Summary

In [10]:
for subject in LIST_SUBJECT:
    metrics_df = pd.read_csv(RESULT_DIR / subject / "per_year_metrics.csv")
    trends_df = pd.read_csv(RESULT_DIR / subject / "topic_trends.csv")

    g = len(trends_df[trends_df["trend"] == "GROWING"])
    s = len(trends_df[trends_df["trend"] == "STABLE"])
    d = len(trends_df[trends_df["trend"] == "DECLINING"])

    summary = {"subject": subject, "num_topics": all_models[subject].k,
               "num_years": len(metrics_df),
               "coherence_mean": round(metrics_df["coherence_cv"].mean(), 6),
               "coherence_std": round(metrics_df["coherence_cv"].std(), 6),
               "irbo_mean": round(metrics_df["irbo_mean"].mean(), 6),
               "irbo_std": round(metrics_df["irbo_mean"].std(), 6),
               "quality_mean": round(metrics_df["topic_quality"].mean(), 6),
               "quality_std": round(metrics_df["topic_quality"].std(), 6),
               "topics_growing": g, "topics_stable": s, "topics_declining": d}

    pd.DataFrame([summary]).to_csv(RESULT_DIR / subject / "evolution_summary.csv", index=False)
    print(f"  {subject.upper()}: C={summary['coherence_mean']:.4f}, "
          f"IRBO={summary['irbo_mean']:.4f}, Q={summary['quality_mean']:.4f} | "
          f"↑{g} →{s} ↓{d}")

  CS: C=0.3421, IRBO=0.8559, Q=0.4850 | ↑6 →39 ↓5
  MATH: C=0.3740, IRBO=0.8872, Q=0.5225 | ↑7 →33 ↓10
  PHYSICS: C=0.3611, IRBO=0.8923, Q=0.5114 | ↑4 →46 ↓10
